####Paso 1 - Leer el archivo JSON usando "DataFrameReader de Spark"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
countries_schema = "countryId INT, countryIsoCode STRING, countryName String"

In [0]:
countries_df = spark.read \
  .schema(countries_schema) \
  .json(f"{bronze_folder_path}/{v_file_date}/country.json")

####Paso 2 - Eliminar columnas no deseadas del DataFrame

In [0]:
from pyspark.sql.functions import col

In [0]:
#countries_dropped_df = countries_df.drop("countryIsoCode")
countries_dropped_df = countries_df.drop(col("countryIsoCode"))

####Paso 3 - Cambiar el nombre de las columnas y añadir "ingestion_date" y "enviroment" 

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
countries_final_df = countries_dropped_df \
    .withColumnRenamed("countryId", "country_id") \
    .withColumnRenamed("countryName", "country_name") \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

####Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#countries_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/countries")
countries_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.countries")

In [0]:
%sql
SELECT *
FROM movie_silver.countries;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/countries

In [0]:
#countries_df.printSchema()

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/countries"))

In [0]:
dbutils.notebook.exit("Success")